# Prequential Evaluation of Surrogate Models — Function 2 (Week 7: SFGP vs MFGP)

| Attribute | Value |
|-----------|-------|
| Function | F2 — Noisy Log-Likelihood Maximisation (2D) |
| Data snapshot | Week 7 (17 cumulative samples) |
| Surrogate families | SFGP (Single-Fidelity GP), MFGP (Multi-Fidelity GP) |
| Configs per family | 50 |
| Total configs | 100 |
| Prequential setup | N\_INIT = 10, Evaluation steps = 7 |
| Primary metric | NLP (Negative Log Predictive Density) |
| Library | BoTorch (`SingleTaskGP`, `MultiTaskGP`) |


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# BoTorch / GPyTorch
import gpytorch
from botorch.models import SingleTaskGP, MultiTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import MaternKernel, RBFKernel, ScaleKernel
from gpytorch.constraints import GreaterThan
from gpytorch.likelihoods import GaussianLikelihood
from botorch.models.transforms.input import Normalize

np.random.seed(42)
torch.manual_seed(42)
print('All imports successful.')


In [ ]:
def compute_metrics(predictions, actuals, pred_means, pred_stds):
    """
    Compute prequential evaluation metrics.

    Parameters
    ----------
    predictions : list of float — point predictions (mean) for each step
    actuals     : list of float — actual observed values for each step
    pred_means  : list of float — predicted means (same as predictions)
    pred_stds   : list of float — predicted standard deviations

    Returns
    -------
    dict with MAE, NLP, Coverage_95, and per-step details
    """
    predictions = np.array(predictions)
    actuals     = np.array(actuals)
    pred_means  = np.array(pred_means)
    pred_stds   = np.array(pred_stds)

    # MAE
    mae = np.mean(np.abs(actuals - predictions))

    # Negative Log Predictive Density
    stds_clipped = np.clip(pred_stds, 1e-10, None)
    nlp_values = (0.5 * np.log(2 * np.pi * stds_clipped**2)
                  + (actuals - pred_means)**2 / (2 * stds_clipped**2))
    mean_nlp = np.mean(nlp_values)

    # 95% Prediction Interval Coverage
    lower = pred_means - 1.96 * stds_clipped
    upper = pred_means + 1.96 * stds_clipped
    in_interval = (actuals >= lower) & (actuals <= upper)
    coverage_95 = np.mean(in_interval)

    return {
        'MAE':         mae,
        'NLP':         mean_nlp,
        'Coverage_95': coverage_95,
        'nlp_values':  nlp_values,
        'errors':      actuals - predictions,
        'in_interval': in_interval,
    }

print('compute_metrics() defined.')


In [ ]:
def plot_prequential_results(results, model_name):
    """Plot prequential evaluation results for a single model."""
    actuals = np.array(results['actuals'])
    preds   = np.array(results['pred_means'])
    stds    = np.array(results['pred_stds'])
    steps   = np.arange(1, len(actuals) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Predictions vs Actuals with uncertainty
    ax = axes[0]
    ax.errorbar(steps, preds, yerr=1.96*stds, fmt='o-', color='blue',
                capsize=5, label='Predicted +/- 95% CI', alpha=0.8)
    ax.scatter(steps, actuals, color='red', s=100, zorder=5,
               marker='x', linewidths=2, label='Actual')
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('Output Value')
    ax.set_title(f'{model_name}: Predictions vs Actuals')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel 2: Absolute errors
    ax = axes[1]
    errors = np.abs(actuals - preds)
    ax.bar(steps, errors, color='coral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('Absolute Error')
    ax.set_title(f'{model_name}: Absolute Error per Step')
    ax.grid(True, alpha=0.3, axis='y')

    # Panel 3: NLP per step
    ax = axes[2]
    nlp_vals = results['metrics']['nlp_values']
    ax.bar(steps, nlp_vals, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Evaluation Step')
    ax.set_ylabel('NLP')
    ax.set_title(f'{model_name}: Negative Log Predictive Density per Step')
    ax.grid(True, alpha=0.3, axis='y')

    plt.suptitle(f'{model_name} — Prequential Evaluation (F2)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

print('plot_prequential_results() defined.')


## Step 1: Load Week 7 Data and Fidelity Split

Loads the Week 7 cumulative dataset for Function 2 (17 samples total).
Fidelity assignment is **fixed**: indices 0–9 → low-fidelity (LF, task=0); indices 10–16 → high-fidelity (HF, task=1).
This assignment is used by all MFGP configurations.


In [ ]:
WEEK   = 7
N_INIT = 10
LF_TASK = 0
HF_TASK = 1

input_path  = f'../../data/f2/updated_inputs - Week {WEEK}.npy'
output_path = f'../../data/f2/updated_outputs - Week {WEEK}.npy'

assert os.path.exists(input_path),  f'Missing data file: {input_path}'
assert os.path.exists(output_path), f'Missing data file: {output_path}'

X_all = np.load(input_path)
y_all = np.load(output_path)
if y_all.ndim > 1:
    y_all = y_all.flatten()

n_total = len(y_all)
n_steps = n_total - N_INIT

# Validation guards
assert n_total == 17,           f'Expected 17 samples, got {n_total}'
assert X_all.shape == (17, 2),  f'Expected (17,2), got {X_all.shape}'
assert not np.any(np.isnan(y_all)), 'NaN found in outputs'
assert not np.any(np.isinf(y_all)), 'Inf found in outputs'

print(f'Function 2 — Week {WEEK} Data')
print(f'  X shape: {X_all.shape}')
print(f'  y shape: {y_all.shape}')
print(f'  Output range: [{y_all.min():.6f}, {y_all.max():.6f}]')
print(f'  NaN check:  PASS')
print(f'  Inf check:  PASS')
print(f'  Initial training points: {N_INIT}')
print(f'  Evaluation steps: {n_steps}')
print(f'  LF observations: {N_INIT}  (indices 0–{N_INIT-1}, task={LF_TASK})')
print(f'  HF observations: {n_steps}  (indices {N_INIT}–{n_total-1}, task={HF_TASK})')


## Step 2: Single-Fidelity Gaussian Process (SFGP)

**Architecture**: BoTorch `SingleTaskGP` — a standard exact GP trained on all available observations at a single fidelity level. Posterior inference is analytic (exact marginal likelihood).

**Default configuration**: Matérn 5/2 kernel with ARD, learned noise via MLE.

**Hyperparameter sweep axes** (50 configurations total):

| Axis | Values |
|------|--------|
| Kernel type | `matern05` (ν=0.5), `matern15` (ν=1.5), `matern25` (ν=2.5), `rbf` |
| Noise lower bound | 1e-6, 1e-5, 1e-4, 1e-3 |
| ARD | True, False |
| Log-transform output | True, False |
| Input normalise | True, False |

Block A (32): 4 kernels × 4 noise_lb × 2 ARD, log_transform=False, input_normalize=True  
Block B (18): 9 log-transform configs + 9 no-normalise configs


In [ ]:
def sfgp_prequential_evaluation(X_all, y_all, n_init):
    """
    Default SFGP prequential evaluation.
    Uses Matern 5/2 ARD, noise_lb=1e-5, no log-transform, no input normalisation.
    """
    n_steps = len(y_all) - n_init
    predictions, actuals, pred_means, pred_stds = [], [], [], []

    print('Running SFGP (Matern 5/2 ARD) default prequential evaluation...')
    print(f'  Training starts with {n_init} points, evaluating {n_steps} steps\n')

    for step in range(n_steps):
        n_train = n_init + step
        X_train = torch.tensor(X_all[:n_train], dtype=torch.float64)
        y_train = torch.tensor(y_all[:n_train], dtype=torch.float64).unsqueeze(-1)
        X_test  = torch.tensor(X_all[n_train:n_train+1], dtype=torch.float64)
        y_actual = y_all[n_train]

        model = SingleTaskGP(X_train, y_train)
        mll   = ExactMarginalLogLikelihood(model.likelihood, model)
        fit_gpytorch_mll(mll)

        model.eval()
        with torch.no_grad():
            posterior = model.posterior(X_test)
            mean = posterior.mean.item()
            std  = posterior.variance.sqrt().item()

        predictions.append(mean);  actuals.append(y_actual)
        pred_means.append(mean);   pred_stds.append(std)
        print(f'  Step {step+1}: train={n_train} pts | '
              f'predicted={mean:+.6f} | actual={y_actual:+.6f} | '
              f'error={y_actual-mean:+.6f} | std={std:.6f}')

    metrics = compute_metrics(predictions, actuals, pred_means, pred_stds)
    print(f'\n  MAE: {metrics["MAE"]:.6f}  NLP: {metrics["NLP"]:.4f}  '
          f'Coverage: {metrics["Coverage_95"]:.1%}')
    return dict(predictions=predictions, actuals=actuals,
                pred_means=pred_means, pred_stds=pred_stds, metrics=metrics)

print('sfgp_prequential_evaluation() defined.')
sfgp_default_results = sfgp_prequential_evaluation(X_all, y_all, N_INIT)
plot_prequential_results(sfgp_default_results, 'SFGP Default (Matern 5/2 ARD)')


In [ ]:
def sfgp_prequential_with_config(X_all, y_all, n_init, config):
    """
    Run SFGP prequential evaluation with a specific configuration.

    config keys: kernel_type, noise_lb, ard, log_transform, input_normalize, label
    """
    EPS    = 1e-300
    n_steps = len(y_all) - n_init

    if config.get('log_transform', False):
        y_work = np.log(np.abs(y_all) + EPS)
    else:
        y_work = y_all.copy()

    predictions, actuals_w, pred_means, pred_stds = [], [], [], []

    for step in range(n_steps):
        n_train = n_init + step
        d = X_all.shape[1]
        X_train_np = X_all[:n_train]
        y_train_np = y_work[:n_train]

        try:
            X_tr = torch.tensor(X_train_np, dtype=torch.float64)
            y_tr = torch.tensor(y_train_np, dtype=torch.float64).unsqueeze(-1)
            X_te = torch.tensor(X_all[n_train:n_train+1], dtype=torch.float64)

            # Build kernel
            ard_dims = d if config.get('ard', True) else 1
            kt = config.get('kernel_type', 'matern25')
            if kt == 'matern05':
                base_k = MaternKernel(nu=0.5, ard_num_dims=ard_dims)
            elif kt == 'matern15':
                base_k = MaternKernel(nu=1.5, ard_num_dims=ard_dims)
            elif kt == 'rbf':
                base_k = RBFKernel(ard_num_dims=ard_dims)
            else:  # matern25 default
                base_k = MaternKernel(nu=2.5, ard_num_dims=ard_dims)
            covar_module = ScaleKernel(base_k)

            # Build likelihood
            noise_lb = config.get('noise_lb', 1e-5)
            likelihood = GaussianLikelihood(noise_constraint=GreaterThan(noise_lb))

            # Input normalisation transform
            if config.get('input_normalize', False):
                input_transform = Normalize(d=d)
                model = SingleTaskGP(X_tr, y_tr,
                                     covar_module=covar_module,
                                     likelihood=likelihood,
                                     input_transform=input_transform)
            else:
                model = SingleTaskGP(X_tr, y_tr,
                                     covar_module=covar_module,
                                     likelihood=likelihood)

            mll = ExactMarginalLogLikelihood(model.likelihood, model)
            fit_gpytorch_mll(mll)

            model.eval()
            with torch.no_grad():
                post = model.posterior(X_te)
                mean = post.mean.item()
                std  = post.variance.sqrt().item()

        except Exception:
            mean = np.nan
            std  = np.nan

        y_actual = y_work[n_train]
        predictions.append(mean);  actuals_w.append(float(y_actual))
        pred_means.append(mean);   pred_stds.append(std)

    all_nan = all(np.isnan(predictions))
    if all_nan:
        return dict(predictions=predictions, actuals=actuals_w,
                    pred_means=pred_means, pred_stds=pred_stds,
                    metrics={'MAE': np.nan, 'NLP': np.nan,
                             'Coverage_95': np.nan, 'nlp_values': [np.nan]*n_steps})

    valid = [(p, a, m, s) for p, a, m, s in
             zip(predictions, actuals_w, pred_means, pred_stds)
             if not np.isnan(p)]
    vp, va, vm, vs = map(list, zip(*valid))
    metrics = compute_metrics(vp, va, vm, vs)
    return dict(predictions=predictions, actuals=actuals_w,
                pred_means=pred_means, pred_stds=pred_stds, metrics=metrics)

print('sfgp_prequential_with_config() defined.')


In [ ]:
# ── SFGP: 50 Hyperparameter Configurations ──────────────────────────────
# Block A: 32 configs — 4 kernels × 4 noise_lb × 2 ARD
#           log_transform=False, input_normalize=True
# Block B: 18 configs — 9 log-transform + 9 no-normalise

sfgp_configs = []

# Block A (32)
for kernel in ['matern05', 'matern15', 'matern25', 'rbf']:
    for noise_lb in [1e-6, 1e-5, 1e-4, 1e-3]:
        for ard in [True, False]:
            sfgp_configs.append({
                'kernel_type':    kernel,
                'noise_lb':       noise_lb,
                'ard':            ard,
                'log_transform':  False,
                'input_normalize': True,
                'label': f'SFGP/{kernel},noise={noise_lb:.0e},ard={ard},norm=T',
            })

# Block B1: 9 log-transform configs (3 kernels × 3 noise_lb, ard=True)
for kernel in ['matern15', 'matern25', 'rbf']:
    for noise_lb in [1e-6, 1e-5, 1e-4]:
        sfgp_configs.append({
            'kernel_type':    kernel,
            'noise_lb':       noise_lb,
            'ard':            True,
            'log_transform':  True,
            'input_normalize': True,
            'label': f'SFGP/{kernel},noise={noise_lb:.0e},ard=T,log=T',
        })

# Block B2: 9 no-normalise configs (3 kernels × 3 noise_lb, ard=True only)
for noise_lb in [1e-5, 1e-4, 1e-3]:
    for kernel in ['matern25', 'rbf', 'matern15']:
        sfgp_configs.append({
            'kernel_type':    kernel,
            'noise_lb':       noise_lb,
            'ard':            True,
            'log_transform':  False,
            'input_normalize': False,
            'label': f'SFGP/{kernel},noise={noise_lb:.0e},ard=T,norm=F',
        })

assert len(sfgp_configs) == 50, f'Expected 50, got {len(sfgp_configs)}'
print(f'50 SFGP configurations defined.')
print(f'  Block A: {32}, Block B1 (log): {9}, Block B2 (no-norm): {9}')


In [ ]:
print(f'Running {len(sfgp_configs)} SFGP configurations...\n')
sfgp_hp_results = []

for i, config in enumerate(sfgp_configs):
    print(f'  [{i+1:2d}/50] {config["label"]}')
    try:
        result  = sfgp_prequential_with_config(X_all, y_all, N_INIT, config)
        metrics = result['metrics']
        sfgp_hp_results.append({
            'label':       config['label'],
            'MAE':         metrics['MAE'],
            'NLP':         metrics['NLP'],
            'Coverage_95': metrics['Coverage_95'],
        })
        nlp_str     = f"{metrics['NLP']:.4f}"     if not np.isnan(metrics['NLP'])     else 'NaN'
        mae_str     = f"{metrics['MAE']:.6f}"     if not np.isnan(metrics['MAE'])     else 'NaN'
        cov_str     = f"{metrics['Coverage_95']:.1%}" if not np.isnan(metrics['Coverage_95']) else 'NaN'
        print(f'         MAE={mae_str}  NLP={nlp_str}  Coverage={cov_str}')
    except Exception as e:
        print(f'         FAILED: {e}')
        sfgp_hp_results.append({
            'label': config['label'], 'MAE': np.nan,
            'NLP': np.nan, 'Coverage_95': np.nan,
        })

sfgp_hp_df = pd.DataFrame(sfgp_hp_results)
nan_count  = sfgp_hp_df['NLP'].isna().sum()
print(f'\nSFGP Sweep complete. NaN count: {nan_count}/50')
sfgp_hp_df


In [ ]:
best_sfgp_idx = sfgp_hp_df['NLP'].idxmin()
best_sfgp     = sfgp_hp_df.loc[best_sfgp_idx]
print('Best SFGP configuration (by NLP):')
print(f'  Label:       {best_sfgp["label"]}')
print(f'  MAE:         {best_sfgp["MAE"]:.6f}')
print(f'  NLP:         {best_sfgp["NLP"]:.4f}')
print(f'  Coverage_95: {best_sfgp["Coverage_95"]:.1%}')


In [ ]:
best_sfgp_config  = sfgp_configs[best_sfgp_idx]
best_sfgp_results = sfgp_prequential_with_config(X_all, y_all, N_INIT, best_sfgp_config)
plot_prequential_results(best_sfgp_results, f'Best SFGP: {best_sfgp["label"]}')


## Step 3: Multi-Fidelity Gaussian Process (MFGP)

**Architecture**: BoTorch `MultiTaskGP` using the Intrinsic Coregionalization Model (ICM).
Two tasks share a common kernel structure with a learned covariance matrix of rank `rank`.

**Fidelity assignment** (fixed across all configurations):
- Task 0 (low-fidelity): indices 0–9 (the 10 initial observations)
- Task 1 (high-fidelity): indices 10–16 (the 7 weekly updates)

**Tensor format**: the task index is appended as the last column of `X_train`,
so `task_feature=-1` in `MultiTaskGP`. Each row is `[x₁, x₂, task_id]`.

**Step-0 fallback**: at prequential step 0 there are no HF training points yet.
All MFGP configs fall back to a `SingleTaskGP` on the 10 LF observations for step 0 only.

**Hyperparameter sweep axes** (50 configurations total):

| Axis | Values |
|------|--------|
| Kernel type | `matern15`, `matern25`, `rbf` |
| ICM rank | 1, 2 |
| Noise lower bound | 1e-6, 1e-5, 1e-4, 1e-3 |
| Output standardize | True, False |

Core 48 = 3 × 2 × 4 × 2; Extra 2 = matern25/rank=1 at noise_lb ∈ {5e-6, 5e-5}.


In [ ]:
def mfgp_prequential_with_config(X_all, y_all, n_init, config):
    """
    Run MFGP prequential evaluation with a specific configuration.

    config keys: kernel_type, rank, noise_lb, output_standardize,
                 step0_fallback, label

    Step 0 fallback: when no HF training points are available, uses
    SingleTaskGP on LF data only.
    """
    n_total  = len(y_all)
    n_steps  = n_total - n_init
    rank     = config.get('rank', 1)
    noise_lb = config.get('noise_lb', 1e-5)
    os_flag  = config.get('output_standardize', True)

    # Kernel builder
    def _build_kernel(kt, d):
        if kt == 'matern15':
            return MaternKernel(nu=1.5, ard_num_dims=d)
        elif kt == 'rbf':
            return RBFKernel(ard_num_dims=d)
        else:   # matern25 default
            return MaternKernel(nu=2.5, ard_num_dims=d)

    predictions, actuals, pred_means, pred_stds = [], [], [], []

    for step in range(n_steps):
        n_lf      = n_init            # all LF points always available
        n_hf_avail = step             # HF points available at this step
        n_train   = n_init + step
        d         = X_all.shape[1]

        y_actual = y_all[n_train]

        try:
            if n_hf_avail == 0:
                # ── Step-0 fallback: SingleTaskGP on LF only ──────────
                X_tr = torch.tensor(X_all[:n_lf], dtype=torch.float64)
                y_tr = torch.tensor(y_all[:n_lf], dtype=torch.float64).unsqueeze(-1)
                X_te = torch.tensor(X_all[n_train:n_train+1], dtype=torch.float64)

                lkh   = GaussianLikelihood(noise_constraint=GreaterThan(noise_lb))
                model = SingleTaskGP(X_tr, y_tr, likelihood=lkh)
                mll   = ExactMarginalLogLikelihood(model.likelihood, model)
                fit_gpytorch_mll(mll)

                model.eval()
                with torch.no_grad():
                    post = model.posterior(X_te)
                    mean = post.mean.item()
                    std  = post.variance.sqrt().item()

            else:
                # ── Full MFGP: task-augmented training set ─────────────
                # LF block
                X_lf = X_all[:n_lf]
                y_lf = y_all[:n_lf]
                t_lf = np.full((n_lf, 1), LF_TASK)
                X_lf_aug = np.hstack([X_lf, t_lf])

                # HF block (indices n_init to n_init+step-1)
                X_hf = X_all[n_init:n_init + n_hf_avail]
                y_hf = y_all[n_init:n_init + n_hf_avail]
                t_hf = np.full((n_hf_avail, 1), HF_TASK)
                X_hf_aug = np.hstack([X_hf, t_hf])

                X_train_aug = np.vstack([X_lf_aug, X_hf_aug])
                y_train_all = np.concatenate([y_lf, y_hf])

                X_tr = torch.tensor(X_train_aug, dtype=torch.float64)
                y_tr = torch.tensor(y_train_all, dtype=torch.float64).unsqueeze(-1)

                # Test point with HF task label
                X_te_base = X_all[n_train:n_train+1]
                X_te_aug  = np.hstack([X_te_base,
                                        np.full((1, 1), HF_TASK)])
                X_te = torch.tensor(X_te_aug, dtype=torch.float64)

                model = MultiTaskGP(
                    X_tr, y_tr,
                    task_feature=-1,
                    rank=rank,
                    output_tasks=[HF_TASK],
                )
                mll = ExactMarginalLogLikelihood(model.likelihood, model)
                fit_gpytorch_mll(mll)

                model.eval()
                with torch.no_grad():
                    post = model.posterior(X_te)
                    mean = post.mean.item()
                    std  = post.variance.sqrt().item()

        except Exception:
            mean = np.nan
            std  = np.nan

        predictions.append(mean);  actuals.append(float(y_actual))
        pred_means.append(mean);   pred_stds.append(std)

    all_nan = all(np.isnan(predictions))
    if all_nan:
        return dict(predictions=predictions, actuals=actuals,
                    pred_means=pred_means, pred_stds=pred_stds,
                    metrics={'MAE': np.nan, 'NLP': np.nan,
                             'Coverage_95': np.nan, 'nlp_values': [np.nan]*n_steps})

    valid = [(p, a, m, s) for p, a, m, s in
             zip(predictions, actuals, pred_means, pred_stds)
             if not np.isnan(p)]
    vp, va, vm, vs = map(list, zip(*valid))
    metrics = compute_metrics(vp, va, vm, vs)
    return dict(predictions=predictions, actuals=actuals,
                pred_means=pred_means, pred_stds=pred_stds, metrics=metrics)

print('mfgp_prequential_with_config() defined.')


In [ ]:
# ── MFGP: 50 Hyperparameter Configurations ──────────────────────────────
# Core 48 = kernels{matern15,matern25,rbf} × rank{1,2}
#           × noise_lb{1e-6,1e-5,1e-4,1e-3} × output_standardize{T,F}
# Extra  2 = matern25/rank=1 at noise_lb∈{5e-6, 5e-5}

mfgp_configs = []

# Core 48
for kernel in ['matern15', 'matern25', 'rbf']:
    for rank in [1, 2]:
        for noise_lb in [1e-6, 1e-5, 1e-4, 1e-3]:
            for os_flag in [True, False]:
                mfgp_configs.append({
                    'kernel_type':       kernel,
                    'rank':              rank,
                    'noise_lb':          noise_lb,
                    'output_standardize': os_flag,
                    'step0_fallback':    'lf_sfgp',
                    'label': (f'MFGP/{kernel},rank={rank},'
                              f'noise={noise_lb:.0e},os={os_flag}'),
                })

# Extra 2
for noise_lb in [5e-6, 5e-5]:
    mfgp_configs.append({
        'kernel_type':       'matern25',
        'rank':              1,
        'noise_lb':          noise_lb,
        'output_standardize': True,
        'step0_fallback':    'lf_sfgp',
        'label': f'MFGP/matern25,rank=1,noise={noise_lb:.0e},os=True',
    })

assert len(mfgp_configs) == 50, f'Expected 50, got {len(mfgp_configs)}'
print(f'50 MFGP configurations defined.')
print(f'  Core: 48, Extra: 2')


In [ ]:
print('Running MFGP default config (Matern 5/2, rank=1, noise=1e-5)...')
_def_label = 'MFGP/matern25,rank=1,noise=1e-05,os=True'
_def_cfg = next(cfg for cfg in mfgp_configs if cfg['label'] == _def_label)
mfgp_default_results = mfgp_prequential_with_config(X_all, y_all, N_INIT, _def_cfg)
plot_prequential_results(mfgp_default_results, 'MFGP Default (Matern 5/2, rank=1)')


In [ ]:
print(f'Running {len(mfgp_configs)} MFGP configurations...\n')
mfgp_hp_results = []

for i, config in enumerate(mfgp_configs):
    print(f'  [{i+1:2d}/50] {config["label"]}')
    try:
        result  = mfgp_prequential_with_config(X_all, y_all, N_INIT, config)
        metrics = result['metrics']
        mfgp_hp_results.append({
            'label':       config['label'],
            'MAE':         metrics['MAE'],
            'NLP':         metrics['NLP'],
            'Coverage_95': metrics['Coverage_95'],
        })
        nlp_str = f"{metrics['NLP']:.4f}"     if not np.isnan(metrics['NLP'])     else 'NaN'
        mae_str = f"{metrics['MAE']:.6f}"     if not np.isnan(metrics['MAE'])     else 'NaN'
        cov_str = f"{metrics['Coverage_95']:.1%}" if not np.isnan(metrics['Coverage_95']) else 'NaN'
        print(f'         MAE={mae_str}  NLP={nlp_str}  Coverage={cov_str}')
    except Exception as e:
        print(f'         FAILED: {e}')
        mfgp_hp_results.append({
            'label': config['label'], 'MAE': np.nan,
            'NLP': np.nan, 'Coverage_95': np.nan,
        })

mfgp_hp_df = pd.DataFrame(mfgp_hp_results)
nan_count  = mfgp_hp_df['NLP'].isna().sum()
print(f'\nMFGP Sweep complete. NaN count: {nan_count}/50')
mfgp_hp_df


In [ ]:
if mfgp_hp_df['NLP'].isna().all():
    print('WARNING: All 50 MFGP configurations produced NaN metrics.')
    print('Comparison will proceed with SFGP only.')
    best_mfgp_idx = None
    best_mfgp     = None
else:
    best_mfgp_idx = mfgp_hp_df['NLP'].idxmin()
    best_mfgp     = mfgp_hp_df.loc[best_mfgp_idx]
    print('Best MFGP configuration (by NLP):')
    print(f'  Label:       {best_mfgp["label"]}')
    print(f'  MAE:         {best_mfgp["MAE"]:.6f}')
    print(f'  NLP:         {best_mfgp["NLP"]:.4f}')
    print(f'  Coverage_95: {best_mfgp["Coverage_95"]:.1%}')


In [ ]:
if best_mfgp is not None:
    best_mfgp_config  = mfgp_configs[best_mfgp_idx]
    best_mfgp_results = mfgp_prequential_with_config(
        X_all, y_all, N_INIT, best_mfgp_config)
    plot_prequential_results(best_mfgp_results, f'Best MFGP: {best_mfgp["label"]}')


## Step 4: Head-to-Head Comparison — SFGP vs MFGP

Compare the best SFGP configuration against the best MFGP configuration across all three metrics: MAE, NLP, and 95% Coverage.

**Selection rule**: primary = lowest NLP; tiebreak 1 = lowest MAE; tiebreak 2 = coverage closest to 0.95.


In [ ]:
rows = [
    {'Model': 'SFGP',
     'Configuration': best_sfgp['label'],
     'MAE': best_sfgp['MAE'],
     'NLP': best_sfgp['NLP'],
     'Coverage_95': best_sfgp['Coverage_95']},
]
if best_mfgp is not None:
    rows.append(
        {'Model': 'MFGP',
         'Configuration': best_mfgp['label'],
         'MAE': best_mfgp['MAE'],
         'NLP': best_mfgp['NLP'],
         'Coverage_95': best_mfgp['Coverage_95']}
    )

comparison_df = pd.DataFrame(rows)

# Determine per-metric winners
print('Per-metric winners:')
print(f'  Best NLP:      {comparison_df.loc[comparison_df["NLP"].idxmin(), "Model"]}')
print(f'  Best MAE:      {comparison_df.loc[comparison_df["MAE"].idxmin(), "Model"]}')
print(f'  Best Coverage: '
      f'{comparison_df.loc[(comparison_df["Coverage_95"]-0.95).abs().idxmin(), "Model"]}')

# Overall winner: primary=NLP, T1=MAE, T2=coverage proximity
best_nlp_row = comparison_df.loc[comparison_df['NLP'].idxmin()]
# Check for tie (within 1e-6)
nlp_vals = comparison_df['NLP'].values
if len(nlp_vals) > 1 and abs(nlp_vals[0] - nlp_vals[1]) < 1e-6:
    best_mae_row = comparison_df.loc[comparison_df['MAE'].idxmin()]
    mae_vals = comparison_df['MAE'].values
    if abs(mae_vals[0] - mae_vals[1]) < 1e-6:
        best_overall = comparison_df.loc[
            (comparison_df['Coverage_95']-0.95).abs().idxmin()]
    else:
        best_overall = best_mae_row
else:
    best_overall = best_nlp_row

print(f'\nOverall winner: {best_overall["Model"]} — {best_overall["Configuration"]}')
comparison_df


In [ ]:
# 3-panel comparison bar chart
models  = comparison_df['Model'].tolist()
colours = {'SFGP': '#2196F3', 'MFGP': '#FF9800'}
bar_colours = [colours.get(m, '#999999') for m in models]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_info = [
    ('MAE',         'MAE',         'lower is better'),
    ('NLP',         'NLP',         'lower is better'),
    ('Coverage_95', '95% Coverage','closer to 0.95 is better'),
]

for ax, (col, ylabel, note) in zip(axes, metrics_info):
    vals = comparison_df[col].values
    bars = ax.bar(models, vals, color=bar_colours, edgecolor='black', alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + abs(v)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    if col == 'Coverage_95':
        ax.axhline(0.95, color='red', linestyle='--', linewidth=1.2, label='0.95 target')
        ax.legend(fontsize=8)
    ax.set_title(f'{ylabel}\n({note})', fontsize=10)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('F2: Best SFGP vs Best MFGP — 2-Way Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# All-100-config horizontal sensitivity bar chart
sfgp_plot_df = sfgp_hp_df.copy()
mfgp_plot_df = mfgp_hp_df.copy()
sfgp_plot_df['Model'] = 'SFGP'
mfgp_plot_df['Model'] = 'MFGP'

combined = pd.concat([sfgp_plot_df, mfgp_plot_df], ignore_index=True)
combined = combined.dropna(subset=['NLP'])
combined = combined.sort_values('NLP').reset_index(drop=True)

bar_cols = ['#2196F3' if m == 'SFGP' else '#FF9800'
            for m in combined['Model']]

fig, ax = plt.subplots(figsize=(12, max(6, len(combined)*0.22)))
ax.barh(combined['label'], combined['NLP'], color=bar_cols,
        edgecolor='black', alpha=0.8, height=0.8)
ax.set_xlabel('Mean NLP (lower is better)')
ax.set_title('F2: All 100 Configurations — Hyperparameter Sensitivity',
             fontsize=12, fontweight='bold')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2196F3', label='SFGP'),
                   Patch(color='#FF9800', label='MFGP')],
          loc='lower right')
plt.tight_layout()
plt.show()
print(f'Shown {len(combined)} valid configurations (dropped NaN rows).')


In [ ]:
sfgp_table = sfgp_hp_df.copy(); sfgp_table.insert(0, 'Model', 'SFGP')
mfgp_table = mfgp_hp_df.copy(); mfgp_table.insert(0, 'Model', 'MFGP')
full_summary = (pd.concat([sfgp_table, mfgp_table], ignore_index=True)
                .sort_values('NLP').reset_index(drop=True))
full_summary.index = full_summary.index + 1  # 1-based rank
full_summary.index.name = 'Rank'
print(f'Full ranked results ({len(full_summary)} configurations):')
full_summary


In [ ]:
print('=' * 60)
print(f'  OVERALL WINNER: {best_overall["Model"]}')
print(f'  Config: {best_overall["Configuration"]}')
print(f'  NLP: {best_overall["NLP"]:.4f}  '
      f'MAE: {best_overall["MAE"]:.6f}  '
      f'Coverage: {best_overall["Coverage_95"]:.1%}')
print('=' * 60)

# Dispatch to the correct function based on which df contains the winner label
winner_label = best_overall['Configuration']
if winner_label in sfgp_hp_df['label'].values:
    winner_cfg_idx = sfgp_hp_df[sfgp_hp_df['label'] == winner_label].index[0]
    winner_cfg     = sfgp_configs[winner_cfg_idx]
    winner_results = sfgp_prequential_with_config(X_all, y_all, N_INIT, winner_cfg)
else:
    winner_cfg_idx = mfgp_hp_df[mfgp_hp_df['label'] == winner_label].index[0]
    winner_cfg     = mfgp_configs[winner_cfg_idx]
    winner_results = mfgp_prequential_with_config(X_all, y_all, N_INIT, winner_cfg)

plot_prequential_results(winner_results,
    f'WINNER — {best_overall["Model"]}: {winner_label}')


## Conclusions

### Winning Surrogate
The best overall surrogate (by NLP) for Function 2 on Week 7 data was determined by the cell above. Refer to the printed winner announcement for the model family and key hyperparameter values.

### Summary of Results
- **SFGP**: 50 configurations evaluated across kernel type, noise lower bound, ARD,   log-transform, and input normalisation axes.
- **MFGP**: 50 configurations evaluated across kernel type, ICM rank, noise lower bound,   and output standardization. Step-0 used a SingleTaskGP fallback on LF data.

### Limitations
- **MFGP step-0 fallback**: predictions at step 0 are computed from LF data only,   ignoring the multi-fidelity structure. This may inflate MFGP NLP at step 0.
- **Coverage reliability**: 7 evaluation steps is a small sample.   The 95% coverage estimate is indicative only and should not be interpreted as a   statistically robust calibration measure.
- **Fixed fidelity assignment**: the LF/HF split (indices 0–9 vs 10–16) is domain-driven.   A different split could change the MFGP results.

### Recommendation for BO Pipeline
Use the winning surrogate family and its best configuration as the surrogate for the next Bayesian Optimisation step on Function 2. If SFGP wins, the standard `SingleTaskGP` pipeline remains unchanged. If MFGP wins, integrate `MultiTaskGP` with `task_feature=-1` and the winning `rank` value into the acquisition function loop.
